Está dando erro ao tentar ler os valores da coluna NET_CO2_1stOrder. Portanto, por enquanto essa colunas está com todos os valores null

In [6]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    dbname="ProjetoMC536",
    user="postgres",
    password="GSW30_curry",
    host="localhost",
    port="5432"
)

cursor = conn.cursor()

In [ ]:
comando = """
SELECT * FROM public."RegiaoAdministrativa"
WHERE nome = 'Amazônia Legal'
"""
cursor.execute(comando)
resultado = cursor.fetchall()
id = resultado[0][0]
print(id)

[(1, 'Amazônia Legal')]
1


In [21]:
# Leitura do CSV
df = pd.read_excel('inpe_EM_BRAmz_results.xlsx', sheet_name='Sem degracação', header=10)


# Seleciona e renomeia as colunas que serão usadas
df = df.rename(columns={
    'Year': 'ano',
    'VR_CO2_1stOrder': 'co2_1a_ordem',
    'VR_CO2_2ndOrder': 'co2_2a_ordem',
    'VR_CO2_2ndOrderFire': 'co2_por_fogo',
    'VR_CO2_2ndOrderDecay': 'co2_por_decaimento',
    'VR_CH42Eq_2ndOrderFire': 'ch4_eq_fogo',
    'VR_N2OEq_2ndOrderFire': 'n2o_eq_fogo',
    'NET_CO2_1stOrder': 'net_co2_1a_ordem',
    'NET_CO2_2ndOrder': 'net_co2_2a_ordem',
    'SV_AreaTotal': 'area_total_secundaria',
    'SV_AreaCleared': 'area_cortada',
    'SV_CO2Emission': 'co2_emitido_secundaria',
    'SV_CO2Absorption': 'co2_absorvido_secundaria',

})
# cont = 0
# for _, row in df.iterrows():
#     print(row)
#     cont += 1
#     if cont > 2:
#         break

df = df[['ano', 'co2_1a_ordem', 'co2_2a_ordem', 'co2_por_fogo', 'co2_por_decaimento',
           'ch4_eq_fogo', 'n2o_eq_fogo', 'net_co2_2a_ordem',
           'area_total_secundaria', 'area_cortada', 'co2_emitido_secundaria',
           'co2_absorvido_secundaria']]
df['IdRegiaoAdministrativa'] = id

# cont = 0
# for _, row in df.iterrows():
#     print(row)
#     cont += 1
#     if cont > 2:
#         break


data = list(df.itertuples(index=False, name=None))

In [ ]:
from psycopg2.extras import execute_values
comando = """
    INSERT INTO public."RelatorioEmissaoGasesAnual"
    (ano, co2_1a_ordem, co2_2a_ordem, co2_por_fogo, co2_por_decaimento,
    ch4_eq_fogo, n2o_eq_fogo, net_co2_2a_ordem, area_total_secundaria, area_cortada, co2_emitido_secundaria,
    co2_absorvido_secundaria, "IdRegiaoAdministrativa")
    VALUES %s
"""
execute_values(cursor, comando, data)

conn.commit()
cursor.close()
conn.close()


In [ ]:
#Caso comando dê erro, desfaz as alterações
conn.rollback()